In [1]:
import os.path

import matplotlib.pyplot as plt
import mne.io
import pandas as pd
import yasa
from yasa import stft_power
import dataset as ds
import numpy as np
import seaborn as sns
from lspopt import spectrogram_lspopt

In [2]:

mne.set_log_level('CRITICAL')

sns.set_theme(
    context='notebook',
    style='whitegrid',
    palette='deep',
    font='Arial',
    font_scale=1.5,
    color_codes=True,
)

plt.rc('axes', unicode_minus=False)
plt.rc('xtick', direction='out')
plt.rc('ytick', direction='out')

In [4]:

path = os.path.join(ds.path['fig'], 'overview')

df_sessions = pd.read_excel(os.path.join(ds.path['tbl'], 'sessions.xlsx'))
animals = df_sessions['animal_id'].unique()
df_psd = pd.DataFrame()

for animal in animals[:1]:

    sessions = df_sessions.query('animal_id == @animal')['session'].unique()
    genotype = df_sessions.query('animal_id == @animal')['genotype'].iloc[0]

    for session in sessions[:2]:

        fname = os.path.join(ds.path['tmp'], 'prep', 'annot_over_loco', f'{animal}_{session}_epochs.fif')
        epochs = mne.read_epochs(fname)
        if epochs.selection.size > 0:
            # prep data
            fname = os.path.join(ds.path['tmp'], 'prep', 'annot_over_loco', f'{animal}_{session}_epochs.fif')
            epochs = mne.read_epochs(fname)
            epoch_length = epochs.tmax + 1 / epochs.info['sfreq']

            df_psd_prep = epochs.compute_psd(picks=[0], fmax=45, method='welch',
                                             n_fft=int(2 * epochs.info['sfreq'])).to_data_frame()
            df_psd_prep = df_psd_prep.rename(columns={epochs.ch_names[0]: 'power'})
            df_psd_prep['power'] = 10 * np.log10(df_psd_prep['power']) + 120
            df_psd_prep['animal'] = animal
            df_psd_prep['session'] = session
            df_psd_prep['genotype'] = genotype
            
            df_psd = pd.concat([df_psd, df_psd_prep], axis=0)
            
df_psd.reset_index(drop=True)

,condition,epoch,freq,power,animal,session,genotype
0,0,382,0.0,-1.297773,0628#,2024-03-14,WT
1,0,382,0.5,4.943026,0628#,2024-03-14,WT
2,0,382,1.0,4.799433,0628#,2024-03-14,WT
3,0,382,1.5,8.839510,0628#,2024-03-14,WT
4,0,382,2.0,3.023788,0628#,2024-03-14,WT
...,...,...,...,...,...,...,...
9641,0,1679,43.0,-11.830474,0628#,2024-04-07,WT
9642,0,1679,43.5,-6.744784,0628#,2024-04-07,WT
9643,0,1679,44.0,-5.311958,0628#,2024-04-07,WT
9644,0,1679,44.5,-13.534411,0628#,2024-04-07,WT


In [5]:
df_psd.to_excel(os.path.join(ds.path['tbl'], 'psd.xlsx'))